In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)

from sklearn.neighbors import KNeighborsRegressor

try:
    from xgboost import XGBRegressor
    xgb_available = True
except:
    xgb_available = False

# LOAD CSV FILES
market_df = pd.read_csv("../data/AAPL_market_features_2021_2024.csv")
error_df = pd.read_csv("../data/error.csv")
news_df = pd.read_csv("../data/news_features.csv")

news_df = news_df[
    ["date", "severity", "sentiment_score", "impact_score", "event_weight"]
]

# DATE FORMAT
market_df["Date"] = pd.to_datetime(market_df["Date"])
error_df["date"] = pd.to_datetime(error_df["date"])
news_df["date"] = pd.to_datetime(news_df["date"])

# MERGE DATASETS
df = (
    market_df
    .merge(error_df, left_on="Date", right_on="date", how="inner")
    .merge(
        news_df[
            [
                "date",
                "sentiment_score",
                "impact_score",
                "event_weight"
            ]
        ],
        on="date",
        how="inner"
    )
)

# Remove duplicate date column
df.drop(columns=["date"], inplace=True)

# Remove unnamed columns if any
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print("Merged Shape :", df.shape)
print(df.head())

# FEATURES
feature_cols = [

    # News Features
    "sentiment_score",
    "impact_score",
    "event_weight",

    # Market Features
    "return_1d",
    "return_5d"

]

X = df[feature_cols]
y = df["error"]

# TRAIN / TEST SPLIT
split = int(len(df) * 0.80)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# MODELS
models = {

    "Linear Regression":
        LinearRegression(),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=300,
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            random_state=42
        ),

    "Extra Trees":
        ExtraTreesRegressor(
            n_estimators=300,
            random_state=42
        ),

    "Hist Gradient Boosting":
        HistGradientBoostingRegressor(
            random_state=42
        ),

    "KNN":
        KNeighborsRegressor(
            n_neighbors=5
        )
}

if xgb_available:
    models["XGBoost"] = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        random_state=42
    )

# TRAIN
results = []

best_model = None
best_score = -999
best_name = ""

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    r2 = r2_score(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))

    results.append([name, r2, mae, rmse])

    if r2 > best_score:
        best_score = r2
        best_model = model
        best_name = name

# RESULTS
results = pd.DataFrame(
    results,
    columns=[
        "Model",
        "R2",
        "MAE",
        "RMSE"
    ]
)

results = results.sort_values(
    "R2",
    ascending=False
)

print("\n==============================")
print(results)
print("==============================")

# SAVE BEST MODEL
joblib.dump(
    best_model,
    "../models/Best_Correction_Model.pkl"
)

print("\nBest Model :", best_name)
print("Best R2    :", round(best_score,4))

# FEATURE IMPORTANCE
if hasattr(best_model, "feature_importances_"):

    importance = pd.DataFrame({

        "Feature": feature_cols,
        "Importance": best_model.feature_importances_

    })

    importance = importance.sort_values(
        "Importance",
        ascending=False
    )

    print("\nFeature Importance")
    print(importance)

    importance.to_csv(
        "Feature_Importance.csv",
        index=False
    )

elif hasattr(best_model, "coef_"):

    coef = pd.DataFrame({

        "Feature": feature_cols,
        "Coefficient": best_model.coef_,
        "Abs_Coefficient": np.abs(best_model.coef_)

    })

    coef = coef.sort_values(
        "Abs_Coefficient",
        ascending=False
    )

    print("\nLinear Regression Coefficients")
    print(coef)

Merged Shape : (972, 25)
        Date  return_1d  return_5d  return_20d  volatility_10  volatility_20  \
0 2021-02-22  -0.029799  -0.069218   -0.092628       0.010266       0.017264   
1 2021-02-23  -0.001111  -0.055034   -0.118052       0.010068       0.015535   
2 2021-02-24  -0.004052  -0.041960   -0.123098       0.010148       0.015436   
3 2021-02-25  -0.034783  -0.067227   -0.147045       0.013064       0.016690   
4 2021-02-26   0.002231  -0.066297   -0.114150       0.013449       0.015532   

    ATR_pct     RSI_14  MACD_12_26_9  MACDh_12_26_9  ...       gap  \
0  0.026330  35.961365     -0.133730      -1.939220  ... -1.809938   
1  0.029198  35.749772     -0.634846      -1.952268  ... -2.179745   
2  0.029291  34.943199     -1.059812      -1.901788  ... -0.895251   
3  0.031674  28.933331     -1.719135      -2.048888  ... -0.651960   
4  0.031620  29.739170     -2.195149      -2.019922  ...  1.556934   

   Momentum_5  Momentum_20  predicted_price  actual_price   error  \
0   